# Meeting Scheduler Agent: Propose Times & Resolve Constraints

**Goal:** build an agent that reads participants' calendars, identifies free slots, proposes meeting times, resolves scheduling constraints, and confirms bookings — all against a **mocked calendar tool**.

---

## Why a Scheduling Agent?

Scheduling a meeting among multiple people is a constraint-satisfaction problem:
- Each person has **busy blocks** (existing meetings, focus time, lunch)
- Meetings have **duration** and **priority** requirements
- Time zones, working hours, and preferences add soft constraints
- The agent must propose the **best available slot**, not just any slot

## What This Notebook Teaches

| Concept | Where it appears |
|---|---|
| **Mocked tool design** | Calendar API with deterministic, inspectable state |
| **Constraint satisfaction** | Finding slots that satisfy all hard constraints |
| **Soft constraint ranking** | Preferring morning vs. afternoon, minimizing fragmentation |
| **Multi-agent coordination** | Checking N calendars simultaneously |
| **Tool call patterns** | Read → reason → propose → confirm |
| **Structured outputs** | Every proposal is a typed dataclass |

```text
Agent Pipeline

  Scheduling Request
       │
       ▼
  ┌──────────────────┐
  │  Parse Request    │  duration, participants, priority, preferences
  └──────┬───────────┘
         │
         ▼
  ┌──────────────────┐
  │  Fetch Calendars  │  read each participant's busy blocks ← TOOL
  └──────┬───────────┘
         │
         ▼
  ┌──────────────────┐
  │  Find Free Slots  │  interval intersection across all calendars
  └──────┬───────────┘
         │
         ▼
  ┌──────────────────┐
  │  Rank & Propose   │  score slots by soft constraints
  └──────┬───────────┘
         │
         ▼
  ┌──────────────────┐
  │  Confirm Booking  │  write to all calendars ← TOOL
  └──────────────────┘
```


## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn

In [ ]:
import json
import re
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, time, timedelta
from enum import Enum
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Project dir:  {PROJECT_DIR}")
print(f"Artifact dir: {ARTIFACT_DIR}")

## 2. Data Model

We define typed structures for everything the agent works with: time blocks, calendar events, scheduling requests, and proposals.


In [ ]:
class Priority(Enum):
    CRITICAL = "critical"     # Must happen today
    HIGH = "high"             # This week
    NORMAL = "normal"         # Flexible
    LOW = "low"               # Optional / can be async

    def __str__(self):
        return self.value


class SlotPreference(Enum):
    MORNING = "morning"       # 09:00–12:00
    AFTERNOON = "afternoon"   # 13:00–17:00
    ANY = "any"

    def __str__(self):
        return self.value


@dataclass
class TimeBlock:
    """A contiguous block of time."""
    start: datetime
    end: datetime
    label: str = ""

    @property
    def duration_minutes(self) -> int:
        return int((self.end - self.start).total_seconds() / 60)

    def overlaps(self, other: "TimeBlock") -> bool:
        return self.start < other.end and other.start < self.end

    def contains(self, dt: datetime) -> bool:
        return self.start <= dt < self.end

    def __repr__(self):
        return (f"TimeBlock({self.start.strftime('%H:%M')}-"
                f"{self.end.strftime('%H:%M')} {self.label!r})")


@dataclass
class CalendarEvent:
    """An event on someone's calendar."""
    event_id: str
    title: str
    start: datetime
    end: datetime
    attendees: list[str] = field(default_factory=list)
    is_recurring: bool = False
    can_reschedule: bool = False

    @property
    def duration_minutes(self) -> int:
        return int((self.end - self.start).total_seconds() / 60)

    def to_block(self) -> TimeBlock:
        return TimeBlock(self.start, self.end, self.title)


@dataclass
class SchedulingRequest:
    """A request to schedule a meeting."""
    title: str
    duration_minutes: int
    participants: list[str]
    target_date: date
    priority: Priority = Priority.NORMAL
    preference: SlotPreference = SlotPreference.ANY
    buffer_minutes: int = 10          # Buffer between meetings
    earliest_hour: int = 9            # Working hours start
    latest_hour: int = 17             # Working hours end
    avoid_lunch: bool = True          # Avoid 12:00-13:00


@dataclass
class SlotProposal:
    """A proposed meeting slot with a score."""
    start: datetime
    end: datetime
    score: float                      # 0.0-1.0
    reasons: list[str]                # Why this score
    conflicts: list[str]              # Any soft conflicts

    @property
    def duration_minutes(self) -> int:
        return int((self.end - self.start).total_seconds() / 60)

    def to_dict(self) -> dict:
        return {
            "start": self.start.isoformat(),
            "end": self.end.isoformat(),
            "duration_minutes": self.duration_minutes,
            "score": round(self.score, 2),
            "reasons": self.reasons,
            "conflicts": self.conflicts,
        }


@dataclass
class BookingResult:
    """Result of a confirmed booking."""
    success: bool
    event_id: str
    title: str
    start: datetime
    end: datetime
    attendees: list[str]
    message: str


print("Data model loaded.")
print(f"  Priority options:    {[p.value for p in Priority]}")
print(f"  Preference options:  {[p.value for p in SlotPreference]}")

## 3. The Mocked Calendar Tool

### Why Mock?

In production, the agent would call Google Calendar API, Microsoft Graph, or Cal.com. Here we build a **fully deterministic, inspectable mock** that:

- Stores events in memory (a Python dict)
- Supports all CRUD operations
- Lets us inspect internal state at any point
- Makes the notebook reproducible without API keys

### Design Principles for Mocked Tools

1. **Same interface as the real tool** — the agent code shouldn't change when swapping to production
2. **Deterministic** — same seed, same results, every time
3. **Inspectable** — we can print the full calendar state for educational clarity
4. **Stateful** — bookings persist across calls within a session


In [ ]:
class MockCalendar:
    """Mocked calendar service supporting multiple users.

    Stores events per user.  Provides the same read/write interface
    a real calendar API would expose.
    """

    def __init__(self):
        self._calendars: dict[str, list[CalendarEvent]] = defaultdict(list)
        self._next_id = 1

    # ── Read operations ──

    def get_events(self, user: str, target_date: date) -> list[CalendarEvent]:
        """Get all events for a user on a specific date."""
        return [
            e for e in self._calendars[user]
            if e.start.date() == target_date
        ]

    def get_busy_blocks(self, user: str, target_date: date) -> list[TimeBlock]:
        """Get busy time blocks for a user on a date."""
        events = self.get_events(user, target_date)
        return [e.to_block() for e in events]

    def is_free(self, user: str, start: datetime, end: datetime) -> bool:
        """Check if a user is free during a time range."""
        query = TimeBlock(start, end)
        for event in self._calendars[user]:
            if event.to_block().overlaps(query):
                return False
        return True

    def get_all_users(self) -> list[str]:
        """List all users with calendars."""
        return list(self._calendars.keys())

    # ── Write operations ──

    def add_event(self, user: str, title: str,
                  start: datetime, end: datetime,
                  attendees: list[str] = None,
                  is_recurring: bool = False,
                  can_reschedule: bool = False) -> CalendarEvent:
        """Add an event to a user's calendar."""
        event = CalendarEvent(
            event_id=f"EVT-{self._next_id:04d}",
            title=title,
            start=start,
            end=end,
            attendees=attendees or [],
            is_recurring=is_recurring,
            can_reschedule=can_reschedule,
        )
        self._calendars[user].append(event)
        self._next_id += 1
        return event

    def book_meeting(self, title: str, start: datetime, end: datetime,
                     participants: list[str]) -> BookingResult:
        """Book a meeting on all participants' calendars.

        This is the agent's write action — it modifies calendar state.
        """
        # Double-check all participants are free
        for user in participants:
            if not self.is_free(user, start, end):
                return BookingResult(
                    success=False, event_id="", title=title,
                    start=start, end=end, attendees=participants,
                    message=f"Conflict: {user} is not free "
                            f"{start.strftime('%H:%M')}-{end.strftime('%H:%M')}",
                )

        # Book on all calendars
        event_id = f"EVT-{self._next_id:04d}"
        for user in participants:
            self.add_event(user, title, start, end, attendees=participants)

        return BookingResult(
            success=True, event_id=event_id, title=title,
            start=start, end=end, attendees=participants,
            message=f"Meeting booked: {title} "
                    f"{start.strftime('%H:%M')}-{end.strftime('%H:%M')} "
                    f"with {', '.join(participants)}",
        )

    # ── Inspection ──

    def print_calendar(self, user: str, target_date: date):
        """Pretty-print a user's calendar for a date."""
        events = sorted(self.get_events(user, target_date),
                        key=lambda e: e.start)
        print(f"  Calendar for {user} on {target_date}:")
        if not events:
            print("    (empty)")
        for e in events:
            print(f"    {e.start.strftime('%H:%M')}-{e.end.strftime('%H:%M')}  "
                  f"{e.title} ({e.duration_minutes}min)")

    def print_all(self, target_date: date):
        """Print all calendars for a date."""
        for user in sorted(self._calendars.keys()):
            self.print_calendar(user, target_date)
            print()


calendar = MockCalendar()
print("MockCalendar initialized.")

## 4. Populating Calendars with Realistic Schedules

We create five team members with typical workday schedules including stand-ups, focus time, lunch, 1:1s, and recurring meetings.


In [ ]:
TARGET_DATE = date(2024, 7, 15)  # A Monday


def dt(hour: int, minute: int = 0) -> datetime:
    """Shorthand for creating datetime on TARGET_DATE."""
    return datetime.combine(TARGET_DATE, time(hour, minute))


# ── Alice: Engineering Manager ──
calendar.add_event("alice", "Team Standup",      dt(9, 0),  dt(9, 15),  ["alice", "bob", "carol"], is_recurring=True)
calendar.add_event("alice", "1:1 with VP Eng",   dt(10, 0), dt(10, 30), ["alice"])
calendar.add_event("alice", "Lunch",             dt(12, 0), dt(13, 0),  ["alice"])
calendar.add_event("alice", "Sprint Planning",   dt(14, 0), dt(15, 0),  ["alice", "bob", "carol", "dave"])
calendar.add_event("alice", "1:1 with Bob",      dt(16, 0), dt(16, 30), ["alice", "bob"])

# ── Bob: Senior Engineer ──
calendar.add_event("bob", "Team Standup",        dt(9, 0),  dt(9, 15),  ["alice", "bob", "carol"], is_recurring=True)
calendar.add_event("bob", "Focus Time",          dt(9, 30), dt(11, 30), ["bob"], can_reschedule=True)
calendar.add_event("bob", "Lunch",               dt(12, 0), dt(12, 45), ["bob"])
calendar.add_event("bob", "Sprint Planning",     dt(14, 0), dt(15, 0),  ["alice", "bob", "carol", "dave"])
calendar.add_event("bob", "1:1 with Alice",      dt(16, 0), dt(16, 30), ["alice", "bob"])

# ── Carol: Frontend Engineer ──
calendar.add_event("carol", "Team Standup",      dt(9, 0),  dt(9, 15),  ["alice", "bob", "carol"], is_recurring=True)
calendar.add_event("carol", "Design Review",     dt(10, 0), dt(11, 0),  ["carol", "eve"])
calendar.add_event("carol", "Lunch",             dt(12, 30), dt(13, 30), ["carol"])
calendar.add_event("carol", "Sprint Planning",   dt(14, 0), dt(15, 0),  ["alice", "bob", "carol", "dave"])
calendar.add_event("carol", "Code Review",       dt(15, 30), dt(16, 0), ["carol", "bob"])

# ── Dave: Backend Engineer ──
calendar.add_event("dave", "Deep Work",          dt(9, 0),  dt(11, 0),  ["dave"], can_reschedule=True)
calendar.add_event("dave", "Lunch",              dt(12, 0), dt(12, 30), ["dave"])
calendar.add_event("dave", "Sprint Planning",    dt(14, 0), dt(15, 0),  ["alice", "bob", "carol", "dave"])
calendar.add_event("dave", "Infra Sync",         dt(15, 0), dt(15, 30), ["dave"])

# ── Eve: Designer ──
calendar.add_event("eve", "Design Review",       dt(10, 0), dt(11, 0),  ["carol", "eve"])
calendar.add_event("eve", "Lunch",               dt(12, 0), dt(13, 0),  ["eve"])
calendar.add_event("eve", "Portfolio Review",    dt(15, 0), dt(16, 0),  ["eve"])

print(f"Calendars populated for {TARGET_DATE}:\n")
calendar.print_all(TARGET_DATE)

## 5. Constraint Solver — Finding Free Slots

This is the core algorithm. It:

1. Collects **busy blocks** from every participant's calendar
2. Merges overlapping busy blocks into a unified timeline
3. Finds **gaps** in the busy timeline that are long enough for the meeting
4. Filters by working hours, lunch avoidance, and buffer requirements

### Hard Constraints vs. Soft Constraints

| Type | Constraint | Effect |
|---|---|---|
| **Hard** | All participants must be free | Slot eliminated if violated |
| **Hard** | Within working hours (9–17) | Slot eliminated |
| **Hard** | Meeting fits in the slot | Slot eliminated if too short |
| **Soft** | Avoid lunch hour (12–13) | Score reduced |
| **Soft** | Prefer morning/afternoon | Score boosted |
| **Soft** | Minimize fragmentation | Score boosted for adjacent slots |
| **Soft** | Buffer between meetings | Score reduced without buffer |


In [ ]:
def merge_blocks(blocks: list[TimeBlock]) -> list[TimeBlock]:
    """Merge overlapping or adjacent time blocks."""
    if not blocks:
        return []
    sorted_blocks = sorted(blocks, key=lambda b: b.start)
    merged = [TimeBlock(sorted_blocks[0].start, sorted_blocks[0].end,
                        sorted_blocks[0].label)]

    for block in sorted_blocks[1:]:
        if block.start <= merged[-1].end:
            # Overlapping or adjacent — extend
            if block.end > merged[-1].end:
                merged[-1] = TimeBlock(merged[-1].start, block.end,
                                       merged[-1].label + " + " + block.label)
        else:
            merged.append(TimeBlock(block.start, block.end, block.label))
    return merged


def find_free_slots(cal: MockCalendar, participants: list[str],
                    target_date: date, duration_minutes: int,
                    earliest_hour: int = 9, latest_hour: int = 17,
                    buffer_minutes: int = 10,
                    avoid_lunch: bool = True) -> list[TimeBlock]:
    """Find all time slots where ALL participants are free.

    Algorithm:
    1. Gather busy blocks from all participants
    2. Add lunch block if avoid_lunch=True
    3. Merge all busy blocks into a unified timeline
    4. Find gaps between merged blocks within working hours
    5. Filter gaps that can fit the meeting + buffer
    """
    # Step 1: Collect all busy blocks
    all_busy: list[TimeBlock] = []
    for user in participants:
        all_busy.extend(cal.get_busy_blocks(user, target_date))

    # Step 2: Add lunch as a busy block (soft, but applied here as hard)
    if avoid_lunch:
        all_busy.append(TimeBlock(
            datetime.combine(target_date, time(12, 0)),
            datetime.combine(target_date, time(13, 0)),
            "Lunch Block",
        ))

    # Step 3: Add working-hours boundaries
    day_start = datetime.combine(target_date, time(earliest_hour, 0))
    day_end = datetime.combine(target_date, time(latest_hour, 0))

    # Step 4: Merge busy blocks
    merged_busy = merge_blocks(all_busy)

    # Step 5: Find gaps
    free_slots = []
    cursor = day_start

    for block in merged_busy:
        if block.start < day_start:
            cursor = max(cursor, block.end)
            continue
        if block.start > day_end:
            break

        gap_start = cursor
        gap_end = min(block.start, day_end)

        if gap_start < gap_end:
            gap = TimeBlock(gap_start, gap_end, "Free")
            if gap.duration_minutes >= duration_minutes:
                free_slots.append(gap)

        cursor = max(cursor, block.end)

    # Check remaining time after last busy block
    if cursor < day_end:
        gap = TimeBlock(cursor, day_end, "Free")
        if gap.duration_minutes >= duration_minutes:
            free_slots.append(gap)

    return free_slots


# Test: find 30-minute slots for alice, bob, carol
request = SchedulingRequest(
    title="Architecture Review",
    duration_minutes=30,
    participants=["alice", "bob", "carol"],
    target_date=TARGET_DATE,
)

free = find_free_slots(
    calendar, request.participants, request.target_date,
    request.duration_minutes, request.earliest_hour, request.latest_hour,
    request.buffer_minutes, request.avoid_lunch,
)

print(f"Free slots for {request.participants} ({request.duration_minutes}min):\n")
for slot in free:
    print(f"  {slot.start.strftime('%H:%M')}-{slot.end.strftime('%H:%M')}  "
          f"({slot.duration_minutes} minutes available)")

## 6. Slot Scoring — Ranking by Soft Constraints

Not all free slots are equal. A good scheduling agent ranks them by **preference signals**.


In [ ]:
def score_slot(slot: TimeBlock, request: SchedulingRequest,
               cal: MockCalendar) -> SlotProposal:
    """Score a free slot based on soft constraints.

    Scoring factors (each contributes to a 0-1 score):
    - Time preference match (morning/afternoon)
    - Distance from lunch hour
    - Buffer availability (gap before/after existing meetings)
    - Meeting density (prefer less fragmented parts of the day)
    - Work-hour centrality (prefer mid-morning over 9:00 or 16:30)
    """
    score = 0.5  # Baseline
    reasons = []
    conflicts = []

    meeting_start = slot.start
    meeting_end = slot.start + timedelta(minutes=request.duration_minutes)

    # ── 1. Time preference ──
    hour = meeting_start.hour
    if request.preference == SlotPreference.MORNING:
        if 9 <= hour < 12:
            score += 0.15
            reasons.append("Matches morning preference")
        else:
            score -= 0.10
            conflicts.append("Outside preferred morning window")
    elif request.preference == SlotPreference.AFTERNOON:
        if 13 <= hour < 17:
            score += 0.15
            reasons.append("Matches afternoon preference")
        else:
            score -= 0.10
            conflicts.append("Outside preferred afternoon window")
    else:
        reasons.append("No time preference — any slot works")

    # ── 2. Distance from lunch ──
    lunch_start = 12
    dist_from_lunch = min(abs(hour - lunch_start), abs((hour + request.duration_minutes / 60) - lunch_start))
    if dist_from_lunch >= 1.5:
        score += 0.05
        reasons.append("Good distance from lunch hour")
    elif dist_from_lunch < 0.5:
        score -= 0.05
        conflicts.append("Too close to lunch hour")

    # ── 3. Work-hour centrality ──
    # Prefer 10:00-11:30 and 14:00-15:30 over edges of the day
    mid_morning = abs(hour - 10.5)
    mid_afternoon = abs(hour - 14.5)
    centrality = min(mid_morning, mid_afternoon)
    if centrality < 1.5:
        score += 0.10
        reasons.append("Central in the workday — good energy time")
    elif hour <= 9 or hour >= 16:
        score -= 0.05
        conflicts.append("At the edge of working hours")

    # ── 4. Buffer check ──
    buffer = timedelta(minutes=request.buffer_minutes)
    for user in request.participants:
        events = cal.get_events(user, request.target_date)
        for evt in events:
            # Check if there's enough buffer before/after
            if evt.end <= meeting_start and (meeting_start - evt.end) < buffer:
                score -= 0.05
                conflicts.append(f"Tight buffer after {user}'s '{evt.title}'")
            if evt.start >= meeting_end and (evt.start - meeting_end) < buffer:
                score -= 0.05
                conflicts.append(f"Tight buffer before {user}'s '{evt.title}'")

    # ── 5. Slot size efficiency ──
    # Prefer using a slot that fits tightly (avoids fragmenting a large gap)
    efficiency = request.duration_minutes / max(slot.duration_minutes, 1)
    if efficiency > 0.7:
        score += 0.05
        reasons.append("Good fit for the available gap")
    elif efficiency < 0.3:
        score += 0.02
        reasons.append("Large gap — slot fits easily")

    # ── Priority boost for critical meetings ──
    if request.priority == Priority.CRITICAL:
        score += 0.05
        reasons.append("Priority boost: critical meeting")

    score = max(0.0, min(1.0, score))

    return SlotProposal(
        start=meeting_start,
        end=meeting_end,
        score=round(score, 2),
        reasons=reasons,
        conflicts=conflicts,
    )


def generate_proposals(cal: MockCalendar,
                       request: SchedulingRequest,
                       max_proposals: int = 5) -> list[SlotProposal]:
    """Generate and rank meeting proposals."""
    free_slots = find_free_slots(
        cal, request.participants, request.target_date,
        request.duration_minutes, request.earliest_hour,
        request.latest_hour, request.buffer_minutes, request.avoid_lunch,
    )

    proposals = []
    for slot in free_slots:
        # Generate proposals at slot boundaries and midpoints
        current = slot.start
        while current + timedelta(minutes=request.duration_minutes) <= slot.end:
            proposal = score_slot(
                TimeBlock(current,
                          current + timedelta(minutes=request.duration_minutes)),
                request, cal,
            )
            proposals.append(proposal)
            current += timedelta(minutes=15)  # 15-min increments

    # Sort by score descending
    proposals.sort(key=lambda p: p.score, reverse=True)
    return proposals[:max_proposals]


# Generate proposals for the test request
proposals = generate_proposals(calendar, request)

print(f"Top proposals for '{request.title}' ({request.duration_minutes}min):")
print(f"Participants: {request.participants}\n")
for i, p in enumerate(proposals, 1):
    print(f"  #{i}  {p.start.strftime('%H:%M')}-{p.end.strftime('%H:%M')}  "
          f"score={p.score:.2f}")
    for r in p.reasons:
        print(f"       + {r}")
    for c in p.conflicts:
        print(f"       - {c}")
    print()

## 7. Assembling the Full Agent

In [ ]:
@dataclass
class AgentResponse:
    """Complete agent output for a scheduling request."""
    request: SchedulingRequest
    proposals: list[SlotProposal]
    selected: Optional[SlotProposal]
    booking: Optional[BookingResult]
    reasoning: str
    success: bool


class MeetingSchedulerAgent:
    """End-to-end meeting scheduling agent."""

    def __init__(self, calendar: MockCalendar):
        self.calendar = calendar
        self._history: list[AgentResponse] = []

    def schedule(self, request: SchedulingRequest,
                 auto_book: bool = False,
                 verbose: bool = False) -> AgentResponse:
        """Process a scheduling request end to end.

        Steps:
        1. Fetch calendars for all participants
        2. Find free slots satisfying hard constraints
        3. Score and rank slots by soft constraints
        4. Optionally book the top-scoring slot

        Args:
            request: The scheduling request
            auto_book: If True, book the top proposal automatically
            verbose: Print step-by-step reasoning
        """
        if verbose:
            print(f"[1] Scheduling: '{request.title}' "
                  f"({request.duration_minutes}min, {request.priority})")
            print(f"    Participants: {request.participants}")
            print(f"    Date: {request.target_date}")
            print(f"    Preference: {request.preference}")

        # Step 1: Fetch calendars (tool call)
        if verbose:
            print(f"\n[2] Fetching calendars...")
            for user in request.participants:
                events = self.calendar.get_events(user, request.target_date)
                print(f"    {user}: {len(events)} event(s)")
                for e in events:
                    print(f"      {e.start.strftime('%H:%M')}-{e.end.strftime('%H:%M')} {e.title}")

        # Step 2-3: Find and score proposals
        proposals = generate_proposals(self.calendar, request)

        if verbose:
            print(f"\n[3] Found {len(proposals)} candidate slot(s)")

        if not proposals:
            response = AgentResponse(
                request=request, proposals=[], selected=None,
                booking=None, success=False,
                reasoning=f"No available slots for {request.duration_minutes}min "
                          f"meeting with {request.participants} on {request.target_date}. "
                          f"All time slots are occupied.",
            )
            self._history.append(response)
            return response

        if verbose:
            print(f"\n[4] Top proposals:")
            for i, p in enumerate(proposals[:3], 1):
                print(f"    #{i} {p.start.strftime('%H:%M')}-{p.end.strftime('%H:%M')} "
                      f"(score={p.score:.2f})")

        # Step 4: Select and optionally book
        selected = proposals[0]
        booking = None

        if auto_book:
            if verbose:
                print(f"\n[5] Auto-booking top slot: "
                      f"{selected.start.strftime('%H:%M')}-{selected.end.strftime('%H:%M')}")

            booking = self.calendar.book_meeting(
                request.title, selected.start, selected.end,
                request.participants,
            )
            if verbose:
                print(f"    Result: {'booked' if booking.success else 'FAILED'}")
                print(f"    {booking.message}")

        reasoning = (
            f"Found {len(proposals)} available slot(s). "
            f"Best: {selected.start.strftime('%H:%M')}-{selected.end.strftime('%H:%M')} "
            f"(score={selected.score:.2f}). "
            + (f"Booked as {booking.event_id}." if booking and booking.success
               else "Awaiting confirmation.")
        )

        response = AgentResponse(
            request=request, proposals=proposals,
            selected=selected, booking=booking,
            reasoning=reasoning,
            success=booking.success if booking else len(proposals) > 0,
        )
        self._history.append(response)
        return response

    @property
    def history(self):
        return self._history


agent = MeetingSchedulerAgent(calendar)
print("MeetingSchedulerAgent ready.")

## 8. Running the Agent — Demo Requests

In [ ]:
demo_requests = [
    SchedulingRequest(
        title="Architecture Review",
        duration_minutes=30,
        participants=["alice", "bob", "carol"],
        target_date=TARGET_DATE,
        priority=Priority.NORMAL,
        preference=SlotPreference.MORNING,
    ),
    SchedulingRequest(
        title="Quick Sync",
        duration_minutes=15,
        participants=["alice", "dave"],
        target_date=TARGET_DATE,
        priority=Priority.HIGH,
        preference=SlotPreference.ANY,
    ),
    SchedulingRequest(
        title="Design Critique",
        duration_minutes=45,
        participants=["carol", "eve"],
        target_date=TARGET_DATE,
        priority=Priority.NORMAL,
        preference=SlotPreference.AFTERNOON,
    ),
    SchedulingRequest(
        title="All-Hands Retrospective",
        duration_minutes=60,
        participants=["alice", "bob", "carol", "dave", "eve"],
        target_date=TARGET_DATE,
        priority=Priority.CRITICAL,
        preference=SlotPreference.ANY,
    ),
    SchedulingRequest(
        title="1:1 Mentoring",
        duration_minutes=30,
        participants=["bob", "eve"],
        target_date=TARGET_DATE,
        priority=Priority.LOW,
        preference=SlotPreference.AFTERNOON,
    ),
]

print("DEMO SCHEDULING RESULTS")
print("=" * 70)
for req in demo_requests:
    resp = agent.schedule(req)
    top = resp.selected
    print(f"\n{'─' * 60}")
    print(f"  {req.title} ({req.duration_minutes}min, {req.priority})")
    print(f"  Participants: {', '.join(req.participants)}")
    if top:
        print(f"  Best slot: {top.start.strftime('%H:%M')}-{top.end.strftime('%H:%M')}  "
              f"score={top.score:.2f}")
        print(f"  Alternatives: {len(resp.proposals) - 1}")
    else:
        print(f"  NO SLOT FOUND")
    print(f"  {resp.reasoning}")

## 9. Verbose Step-by-Step Walkthrough

In [ ]:
# Reset calendar for a clean walkthrough
wt_calendar = MockCalendar()

# Re-add events for alice and bob only
wt_calendar.add_event("alice", "Standup",   dt(9, 0),  dt(9, 15),  ["alice", "bob"])
wt_calendar.add_event("alice", "1:1 VP",    dt(10, 0), dt(10, 30), ["alice"])
wt_calendar.add_event("alice", "Lunch",     dt(12, 0), dt(13, 0),  ["alice"])
wt_calendar.add_event("alice", "Planning",  dt(14, 0), dt(15, 0),  ["alice", "bob"])

wt_calendar.add_event("bob", "Standup",     dt(9, 0),  dt(9, 15),  ["alice", "bob"])
wt_calendar.add_event("bob", "Focus Time",  dt(9, 30), dt(11, 30), ["bob"])
wt_calendar.add_event("bob", "Lunch",       dt(12, 0), dt(12, 45), ["bob"])
wt_calendar.add_event("bob", "Planning",    dt(14, 0), dt(15, 0),  ["alice", "bob"])

wt_agent = MeetingSchedulerAgent(wt_calendar)

print("VERBOSE WALKTHROUGH")
print("=" * 60)

req = SchedulingRequest(
    title="Code Review Session",
    duration_minutes=30,
    participants=["alice", "bob"],
    target_date=TARGET_DATE,
    priority=Priority.NORMAL,
    preference=SlotPreference.MORNING,
)

resp = wt_agent.schedule(req, verbose=True)

print(f"\n{'='*60}")
print(f"FINAL ANSWER: {resp.reasoning}")

## 10. Booking — Writing to the Calendar

So far we've only *proposed* slots. Now we demonstrate the **write action**: booking a meeting on all participants' calendars.


In [ ]:
# Book the top proposal from the walkthrough
print("BEFORE BOOKING:")
wt_calendar.print_all(TARGET_DATE)

req_book = SchedulingRequest(
    title="API Design Discussion",
    duration_minutes=30,
    participants=["alice", "bob"],
    target_date=TARGET_DATE,
    priority=Priority.HIGH,
    preference=SlotPreference.AFTERNOON,
)

resp = wt_agent.schedule(req_book, auto_book=True, verbose=True)

print(f"\n\nAFTER BOOKING:")
wt_calendar.print_all(TARGET_DATE)

## 11. Handling Conflicts & Edge Cases

What happens when there is **no available slot**? The agent detects this and suggests alternatives.


In [ ]:
# Create a fully-booked calendar to force a conflict
conflict_cal = MockCalendar()
# Fill alice's entire day
for h in range(9, 17):
    conflict_cal.add_event("alice", f"Meeting {h}", dt(h, 0), dt(h+1, 0), ["alice"])

# Bob has one free slot
conflict_cal.add_event("bob", "All-day block", dt(9, 0), dt(16, 0), ["bob"])

conflict_agent = MeetingSchedulerAgent(conflict_cal)

conflict_req = SchedulingRequest(
    title="Impossible Meeting",
    duration_minutes=30,
    participants=["alice", "bob"],
    target_date=TARGET_DATE,
)

resp = conflict_agent.schedule(conflict_req, verbose=True)

print(f"\nResult: success={resp.success}")
print(f"Reasoning: {resp.reasoning}")

# ── What if we try a very short meeting? ──
print("\n" + "─" * 50)
print("Trying with only Alice (who's fully booked):\n")

solo_req = SchedulingRequest(
    title="Solo Check-in",
    duration_minutes=15,
    participants=["alice"],
    target_date=TARGET_DATE,
)

resp2 = conflict_agent.schedule(solo_req)
print(f"Result: success={resp2.success}")
print(f"Reasoning: {resp2.reasoning}")

## 12. Multi-Request Cascade — Scheduling Multiple Meetings

When an agent books one meeting, it changes the calendar state. Subsequent scheduling must account for the new event. This is the **cascade effect**.


In [ ]:
cascade_cal = MockCalendar()
cascade_cal.add_event("alice", "Standup",  dt(9, 0),  dt(9, 15))
cascade_cal.add_event("alice", "Lunch",    dt(12, 0), dt(13, 0))
cascade_cal.add_event("bob",   "Standup",  dt(9, 0),  dt(9, 15))
cascade_cal.add_event("bob",   "Lunch",    dt(12, 0), dt(13, 0))

cascade_agent = MeetingSchedulerAgent(cascade_cal)

cascade_requests = [
    SchedulingRequest("Meeting A", 60, ["alice", "bob"], TARGET_DATE,
                      preference=SlotPreference.MORNING),
    SchedulingRequest("Meeting B", 45, ["alice", "bob"], TARGET_DATE,
                      preference=SlotPreference.MORNING),
    SchedulingRequest("Meeting C", 30, ["alice", "bob"], TARGET_DATE,
                      preference=SlotPreference.AFTERNOON),
]

print("CASCADE BOOKING — 3 meetings for alice + bob")
print("=" * 60)

for i, req in enumerate(cascade_requests, 1):
    print(f"\n--- Request {i}: {req.title} ({req.duration_minutes}min) ---")
    resp = cascade_agent.schedule(req, auto_book=True)
    if resp.booking and resp.booking.success:
        print(f"  Booked: {resp.selected.start.strftime('%H:%M')}-"
              f"{resp.selected.end.strftime('%H:%M')}  "
              f"score={resp.selected.score:.2f}")
    else:
        print(f"  FAILED: {resp.reasoning}")

print(f"\n{'='*60}")
print("FINAL CALENDARS:")
cascade_cal.print_all(TARGET_DATE)

## 13. Visualizing Calendars & Proposals

In [ ]:
def plot_calendar_day(cal: MockCalendar, users: list[str],
                      target_date: date, proposals: list[SlotProposal] = None,
                      title: str = "Calendar View"):
    """Visualize a day's calendar as a Gantt-style chart."""
    fig, ax = plt.subplots(figsize=(14, max(3, len(users) * 1.2 + 1)))

    colors = {
        "meeting": "#3498db",
        "focus": "#9b59b6",
        "lunch": "#95a5a6",
        "proposal": "#2ecc71",
    }

    y_labels = []
    y_positions = []

    for i, user in enumerate(users):
        y = len(users) - i - 1
        y_labels.append(user)
        y_positions.append(y)

        events = sorted(cal.get_events(user, target_date), key=lambda e: e.start)
        for evt in events:
            start_h = evt.start.hour + evt.start.minute / 60
            dur_h = evt.duration_minutes / 60

            color = colors["lunch"] if "lunch" in evt.title.lower() else \
                    (colors["focus"] if "focus" in evt.title.lower() or
                     "deep" in evt.title.lower()
                     else colors["meeting"])

            ax.barh(y, dur_h, left=start_h, height=0.6,
                    color=color, alpha=0.7, edgecolor="black", linewidth=0.5)
            if dur_h >= 0.3:
                ax.text(start_h + dur_h / 2, y, evt.title[:12],
                        ha="center", va="center", fontsize=7,
                        color="white", fontweight="bold")

    # Plot proposals
    if proposals:
        for p in proposals[:3]:
            start_h = p.start.hour + p.start.minute / 60
            dur_h = p.duration_minutes / 60
            for y in y_positions:
                ax.barh(y, dur_h, left=start_h, height=0.6,
                        color=colors["proposal"], alpha=0.3,
                        edgecolor="green", linewidth=2, linestyle="--")

    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels, fontsize=10)
    ax.set_xlim(8.5, 17.5)
    ax.set_xlabel("Hour of Day")
    ax.set_title(title, fontsize=12, fontweight="bold")

    # Hour grid
    ax.set_xticks(range(9, 18))
    ax.set_xticklabels([f"{h}:00" for h in range(9, 18)], fontsize=8)
    ax.grid(axis="x", alpha=0.3)

    # Legend
    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor=colors["meeting"], alpha=0.7, label="Meeting"),
        Patch(facecolor=colors["focus"], alpha=0.7, label="Focus Time"),
        Patch(facecolor=colors["lunch"], alpha=0.7, label="Lunch"),
        Patch(facecolor=colors["proposal"], alpha=0.3, edgecolor="green",
              linewidth=2, linestyle="--", label="Proposed Slot"),
    ]
    ax.legend(handles=legend_elems, loc="upper right", fontsize=8)

    plt.tight_layout()
    plt.show()


# Visualize the full team's calendar with proposals
arch_proposals = generate_proposals(calendar, demo_requests[0], max_proposals=3)
plot_calendar_day(
    calendar,
    ["alice", "bob", "carol", "dave", "eve"],
    TARGET_DATE,
    proposals=arch_proposals,
    title=f"Team Calendar — {TARGET_DATE} (proposals for '{demo_requests[0].title}')",
)

In [ ]:
# Proposal score comparison chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: Score comparison across demo requests
ax = axes[0]
req_names = []
top_scores = []
alt_scores = []
for req in demo_requests:
    props = generate_proposals(calendar, req)
    req_names.append(req.title[:18])
    top_scores.append(props[0].score if props else 0)
    alt_scores.append(np.mean([p.score for p in props[1:]]) if len(props) > 1 else 0)

x = np.arange(len(req_names))
ax.bar(x - 0.15, top_scores, 0.3, label="Top proposal", color="#2ecc71", alpha=0.8)
ax.bar(x + 0.15, alt_scores, 0.3, label="Avg alternative", color="#95a5a6", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(req_names, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Score")
ax.set_title("Proposal Scores by Request")
ax.legend(fontsize=8)
ax.set_ylim(0, 1)

# Chart 2: Factor breakdown for top proposal of first request
ax2 = axes[1]
if arch_proposals:
    top = arch_proposals[0]
    factors = top.reasons + top.conflicts
    factor_scores = [0.1 if "+" not in f and "conflict" not in f.lower()
                     else -0.05 for f in factors]
    # Simplified: show reasons vs conflicts counts
    reason_count = len(top.reasons)
    conflict_count = len(top.conflicts)
    ax2.bar(["Positive\nfactors", "Negative\nfactors"],
            [reason_count, conflict_count],
            color=["#2ecc71", "#e74c3c"], alpha=0.8)
    ax2.set_ylabel("Count")
    ax2.set_title(f"Score Breakdown: {top.start.strftime('%H:%M')}-{top.end.strftime('%H:%M')}")

plt.suptitle("Meeting Scheduler Agent — Analytics", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 14. Designing Mocked Tools — Best Practices

### The Mock-First Pattern

Building agents mock-first has several advantages:

1. **Reproducibility** — no network calls, no API keys, same result every time
2. **Inspectability** — print internal state at any step
3. **Speed** — no latency, instant results
4. **Testability** — write assertions against deterministic outputs
5. **Safety** — no risk of modifying real calendars during development

### Interface Parity

The mock must match the production API's **interface**, not its implementation:

```python
# Mock
class MockCalendar:
    def get_events(self, user: str, date: date) -> list[CalendarEvent]: ...
    def is_free(self, user: str, start: datetime, end: datetime) -> bool: ...
    def book_meeting(self, title, start, end, participants) -> BookingResult: ...

# Production (same interface, different backend)
class GoogleCalendar:
    def get_events(self, user: str, date: date) -> list[CalendarEvent]:
        response = google_api.events().list(calendarId=user, ...).execute()
        return [CalendarEvent.from_google(e) for e in response["items"]]

    def is_free(self, user: str, start: datetime, end: datetime) -> bool:
        body = {"timeMin": start.isoformat(), "timeMax": end.isoformat(), ...}
        response = google_api.freebusy().query(body=body).execute()
        return len(response["calendars"][user]["busy"]) == 0

    def book_meeting(self, title, start, end, participants) -> BookingResult:
        event = {"summary": title, "start": {"dateTime": start.isoformat()}, ...}
        result = google_api.events().insert(calendarId="primary", body=event).execute()
        return BookingResult(success=True, event_id=result["id"], ...)
```

### Key design rules

| Rule | Rationale |
|---|---|
| Return typed objects, not dicts | Safer, self-documenting |
| Separate read and write methods | Makes safety analysis easy |
| Include inspection methods | Enables educational transparency |
| Make state resettable | Supports testing and demos |
| Log all mutations | Enables audit and undo |


## 15. Agent Evaluation

In [ ]:
eval_cases = [
    {
        "name": "Simple 2-person meeting finds slots",
        "request": SchedulingRequest("Test A", 30, ["alice", "bob"], TARGET_DATE),
        "expect_success": True,
        "min_proposals": 1,
    },
    {
        "name": "All-hands with 5 people finds at least 1 slot",
        "request": SchedulingRequest("Test B", 30,
                                     ["alice", "bob", "carol", "dave", "eve"],
                                     TARGET_DATE),
        "expect_success": True,
        "min_proposals": 1,
    },
    {
        "name": "Morning preference ranked higher",
        "request": SchedulingRequest("Test C", 30, ["alice", "dave"],
                                     TARGET_DATE, preference=SlotPreference.MORNING),
        "expect_success": True,
        "check_morning": True,
    },
    {
        "name": "Impossible meeting returns no proposals",
        "request": SchedulingRequest("Test D", 480, ["alice"], TARGET_DATE),
        "expect_success": False,
        "min_proposals": 0,
    },
    {
        "name": "Short meeting finds more slots than long meeting",
        "short": SchedulingRequest("Short", 15, ["alice", "bob"], TARGET_DATE),
        "long": SchedulingRequest("Long", 120, ["alice", "bob"], TARGET_DATE),
        "compare_slot_counts": True,
    },
    {
        "name": "Booking changes calendar state",
        "test_booking": True,
    },
    {
        "name": "Proposals are sorted by score descending",
        "request": SchedulingRequest("Test F", 30, ["alice", "bob"], TARGET_DATE),
        "check_sorted": True,
    },
]


eval_results = []
eval_agent = MeetingSchedulerAgent(calendar)

for tc in eval_cases:
    passed = False

    if tc.get("compare_slot_counts"):
        short_props = generate_proposals(calendar, tc["short"])
        long_props = generate_proposals(calendar, tc["long"])
        passed = len(short_props) >= len(long_props)
    elif tc.get("test_booking"):
        book_cal = MockCalendar()
        book_cal.add_event("x", "Lunch", dt(12, 0), dt(13, 0))
        book_agent = MeetingSchedulerAgent(book_cal)
        req = SchedulingRequest("Booking Test", 30, ["x"], TARGET_DATE)
        resp = book_agent.schedule(req, auto_book=True)
        events_after = book_cal.get_events("x", TARGET_DATE)
        passed = resp.booking is not None and resp.booking.success and len(events_after) == 2
    elif tc.get("check_sorted"):
        resp = eval_agent.schedule(tc["request"])
        scores = [p.score for p in resp.proposals]
        passed = scores == sorted(scores, reverse=True)
    elif tc.get("check_morning"):
        resp = eval_agent.schedule(tc["request"])
        if resp.selected:
            passed = resp.selected.start.hour < 12
        else:
            passed = False
    else:
        resp = eval_agent.schedule(tc["request"])
        passed = resp.success == tc.get("expect_success", True)
        if "min_proposals" in tc:
            passed = passed and len(resp.proposals) >= tc["min_proposals"]

    eval_results.append({"test": tc["name"], "passed": passed})

print("EVALUATION RESULTS")
print("=" * 65)
for er in eval_results:
    icon = "pass" if er["passed"] else "FAIL"
    print(f"  [{icon:4s}]  {er['test']}")

pass_rate = sum(1 for e in eval_results if e["passed"]) / len(eval_results)
print(f"\nOverall: {sum(1 for e in eval_results if e['passed'])}/{len(eval_results)} "
      f"({pass_rate:.0%})")

## 16. Scheduling Patterns — Visual Analysis

In [ ]:
# Analyze where proposals tend to land
all_proposal_hours = []
all_proposal_scores = []

for req in demo_requests:
    props = generate_proposals(calendar, req, max_proposals=10)
    for p in props:
        all_proposal_hours.append(p.start.hour + p.start.minute / 60)
        all_proposal_scores.append(p.score)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Chart 1: Distribution of proposed times
ax = axes[0]
ax.hist(all_proposal_hours, bins=np.arange(9, 18, 0.5),
        color="#3498db", alpha=0.7, edgecolor="black")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Frequency")
ax.set_title("When Are Meetings Proposed?")
ax.axvspan(12, 13, alpha=0.15, color="red", label="Lunch")
ax.legend(fontsize=8)

# Chart 2: Score vs. time of day
ax2 = axes[1]
ax2.scatter(all_proposal_hours, all_proposal_scores,
            c=all_proposal_scores, cmap="RdYlGn", s=60,
            alpha=0.7, edgecolors="black", linewidth=0.5)
ax2.set_xlabel("Hour of Day")
ax2.set_ylabel("Score")
ax2.set_title("Score vs. Time of Day")
ax2.axhline(y=np.mean(all_proposal_scores), color="gray",
            linestyle="--", alpha=0.5)

# Chart 3: Meeting duration vs. available slots
ax3 = axes[2]
durations = [15, 30, 45, 60, 90, 120]
slot_counts = []
for dur in durations:
    req = SchedulingRequest("Test", dur, ["alice", "bob", "carol"],
                            TARGET_DATE)
    props = generate_proposals(calendar, req, max_proposals=20)
    slot_counts.append(len(props))

ax3.bar([str(d) for d in durations], slot_counts,
        color="#2ecc71", alpha=0.8, edgecolor="black")
ax3.set_xlabel("Meeting Duration (min)")
ax3.set_ylabel("Available Slots")
ax3.set_title("Slots Available vs. Duration")

plt.suptitle("Scheduling Patterns", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 17. Production Considerations

### Replacing the Mock with Real APIs

| Mock method | Google Calendar API | Microsoft Graph |
|---|---|---|
| `get_events()` | `events().list()` | `GET /me/calendar/events` |
| `is_free()` | `freebusy().query()` | `POST /me/calendar/getSchedule` |
| `book_meeting()` | `events().insert()` | `POST /me/events` |

### Challenges Not Covered Here

| Challenge | Description |
|---|---|
| **Time zones** | Participants in different zones need TZ-aware datetimes |
| **Recurring meetings** | Agent must check the full recurrence series |
| **Tentative events** | Some events are "maybe" — should they block? |
| **Multi-day search** | If today has no slots, try tomorrow, then next week |
| **Rescheduling** | Moving an existing meeting to accommodate a higher-priority one |
| **Delegation** | "Book with Alice's team" — who exactly? |
| **Privacy** | Not all attendees want their event titles visible |
| **Concurrent booking** | Two agents booking the same slot simultaneously |

### LLM Integration

```python
# The LLM interprets natural language, the agent handles scheduling logic
user_msg = "Can you find 30 minutes for Alice and Bob tomorrow morning?"

# LLM parses into a SchedulingRequest
parsed = llm.parse(user_msg)
# → SchedulingRequest(title="Meeting", duration_minutes=30,
#                     participants=["alice", "bob"],
#                     target_date=date(2024, 7, 16),
#                     preference=SlotPreference.MORNING)

# Agent does the constraint solving
response = agent.schedule(parsed)

# LLM formats the response
reply = llm.format_response(response)
# → "I found a slot for Alice and Bob tomorrow at 10:30-11:00.
#    It scores 0.82 — good distance from lunch, matches your
#    morning preference. Shall I book it?"
```


## 18. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "meeting_scheduler_agent",
    "target_date": TARGET_DATE.isoformat(),
    "team_members": calendar.get_all_users(),
    "demo_requests": len(demo_requests),
    "total_proposals_generated": sum(
        len(r.proposals) for r in agent.history
    ),
    "bookings_made": sum(
        1 for r in agent.history if r.booking and r.booking.success
    ),
    "evaluation_pass_rate": pass_rate,
    "scoring_factors": [
        "time_preference", "lunch_distance", "work_hour_centrality",
        "buffer_availability", "slot_efficiency", "priority_boost",
    ],
    "constraint_types": {
        "hard": ["participant_availability", "working_hours",
                 "duration_fit", "lunch_block"],
        "soft": ["morning/afternoon_preference", "buffer_between_meetings",
                 "centrality_in_day", "slot_fragmentation"],
    },
}

log_path = ARTIFACT_DIR / "meeting_scheduler_agent_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")
print(f"Saved: {log_path}")
print(f"\nFinal stats:")
print(f"  Demo requests:    {log['demo_requests']}")
print(f"  Proposals made:   {log['total_proposals_generated']}")
print(f"  Evaluation:       {log['evaluation_pass_rate']:.0%}")

## 19. Key Takeaways

### What We Built
- A **meeting scheduling agent** that reads calendars, solves constraints, scores slots, and books meetings
- A **mocked calendar tool** with the same interface as Google Calendar / Microsoft Graph
- A **constraint solver** that handles hard constraints (availability, working hours) and soft constraints (preferences, buffers, centrality)
- A **slot scorer** that ranks proposals on 6 factors and explains each decision
- **Cascade booking** — scheduling multiple meetings in sequence with state updates

### Constraint Satisfaction Concepts
1. **Hard constraints** eliminate slots entirely (must be met)
2. **Soft constraints** adjust scores (better vs. worse slots)
3. **Greedy scheduling** — book the best available slot, then re-solve for the next request
4. **Fragmentation** — booking a meeting splits a free block, reducing options for later requests

### Mocked Tool Design
1. **Same interface** as the production API — swap without code changes
2. **Deterministic state** — reproducible results for testing and education
3. **Inspectable internals** — print calendar state at any step
4. **Separate read/write** — makes safety analysis straightforward
5. **Typed return values** — `CalendarEvent`, `BookingResult`, not raw dicts

### The Agent Pattern
```
Parse request → Read tools → Reason (constraint solve) → Propose → Confirm → Write tools
```
The agent never writes without first reading, reasoning, and proposing. This is the **read-reason-propose-confirm** pattern that makes agents safe and transparent.
